In [1]:
# ============================================================
# 10_cfe_case_study.ipynb
# Illustrative CFE Case Study
#
# For each disease, select 2 representative declined
# instances (one young, one elderly) and show:
#   - Original feature values
#   - CFE without guardrail
#   - CFE with guardrail
#   - phi constraint status for each CFE
# ============================================================


# ─────────────────────────────────────────────
# Cell 1 | Imports
# ─────────────────────────────────────────────
import os
import warnings
import numpy as np
import pandas as pd
import joblib
import dice_ml

warnings.filterwarnings('ignore')

BASE_OUT = os.path.join('..', 'outputs')
BASE_MOD = os.path.join('..', 'outputs', 'models')

DISEASE_REGISTRY = {
    'htn': {
        'target'      : 'HE_HP',
        'label'       : 'Hypertension',
        'feature_cols': ['BD1_11', 'BD2_1', 'BS3_1',
                         'BE3_75', 'BE3_91', 'pa_aerobic',
                         'L_BR_FQ', 'BH1',
                         'HE_BMI', 'HE_wc', 'HE_wt',
                         'N_NA', 'N_K'],
        'cont_features': ['HE_BMI', 'HE_wc', 'HE_wt',
                          'N_NA', 'N_K'],
    },
    'dm': {
        'target'      : 'HE_DM_HbA1c',
        'label'       : 'Diabetes',
        'feature_cols': ['BD1_11', 'BD2_1', 'BS3_1',
                         'BE3_75', 'BE3_91', 'pa_aerobic',
                         'L_BR_FQ', 'BH1',
                         'HE_BMI', 'HE_wc', 'HE_wt',
                         'N_SUGAR', 'N_CHO', 'N_EN'],
        'cont_features': ['HE_BMI', 'HE_wc', 'HE_wt',
                          'N_SUGAR', 'N_CHO', 'N_EN'],
    },
    'dys': {
        'target'      : 'HE_HCHOL',
        'label'       : 'Dyslipidemia',
        'feature_cols': ['BD1_11', 'BD2_1', 'BS3_1',
                         'BE3_75', 'BE3_91', 'pa_aerobic',
                         'L_BR_FQ', 'BH1',
                         'HE_BMI', 'HE_wc', 'HE_wt',
                         'N_FAT', 'N_CHOL'],
        'cont_features': ['HE_BMI', 'HE_wc', 'HE_wt',
                          'N_FAT', 'N_CHOL'],
    },
}

FEATURE_LABEL = {
    'BD1_11' : 'Alcohol drinking frequency',
    'BD2_1'  : 'Alcohol amount per occasion',
    'BS3_1'  : 'Current smoking',
    'BE3_75' : 'High-intensity PA: leisure',
    'BE3_91' : 'PA: transportation',
    'pa_aerobic': 'Aerobic PA practice',
    'L_BR_FQ': 'Breakfast frequency',
    'BH1'    : 'Health checkup (past year)',
    'HE_BMI' : 'BMI (kg/m²)',
    'HE_wc'  : 'Waist circumference (cm)',
    'HE_wt'  : 'Body weight (kg)',
    'N_NA'   : 'Sodium intake (mg)',
    'N_K'    : 'Potassium intake (mg)',
    'N_SUGAR': 'Sugar intake (g)',
    'N_CHO'  : 'Carbohydrate intake (g)',
    'N_EN'   : 'Energy intake (kcal)',
    'N_FAT'  : 'Fat intake (g)',
    'N_CHOL' : 'Dietary cholesterol (mg)',
}

PHI_COST_BOUNDS = {
    'HE_BMI': 3.0,   'HE_wc': 8.0,   'HE_wt': 8.0,
    'N_NA'  : 1500.0, 'N_K'  : 1000.0,
    'N_SUGAR': 30.0,  'N_CHO': 100.0, 'N_EN': 500.0,
    'N_FAT' : 30.0,   'N_CHOL': 150.0,
}

G1_PERMITTED_RANGE = {
    'HE_BMI': [15.0, 40.0], 'HE_wc': [55.0, 110.0],
    'HE_wt' : [35.0, 95.0],
    'N_NA'  : [200.0, 7000.0], 'N_K': [300.0, 5500.0],
    'N_SUGAR': [0.0, 150.0],  'N_CHO': [50.0, 500.0],
    'N_EN'  : [500.0, 3500.0],
    'N_FAT' : [5.0, 120.0],  'N_CHOL': [0.0, 700.0],
}

RANDOM_SEED = 42

def check_phi_caus(orig, cfe):
    dw = cfe['HE_wt']  - orig['HE_wt']
    db = cfe['HE_BMI'] - orig['HE_BMI']
    if abs(dw) >= 1.0:
        return (dw * db) >= 0
    return abs(db) <= 0.5

def check_phi_cost(orig, cfe, cont_features):
    for f in cont_features:
        if f not in PHI_COST_BOUNDS:
            continue
        if abs(cfe[f] - orig[f]) > PHI_COST_BOUNDS[f]:
            return False
    return True

def check_g1(cfe, cont_features):
    for f in cont_features:
        if f not in G1_PERMITTED_RANGE:
            continue
        lo, hi = G1_PERMITTED_RANGE[f]
        if not (lo <= cfe[f] <= hi):
            return False
    return True

def make_query(row, feature_cols):
    q = row[feature_cols].to_frame().T\
                         .reset_index(drop=True)
    for col in feature_cols:
        q[col] = q[col].astype(float)
    return q

def build_dice(df, feature_cols, target_col, model):
    df_d = df[feature_cols + [target_col]].copy()
    df_d[target_col] = df_d[target_col].astype(int)
    d = dice_ml.Data(dataframe=df_d,
                     continuous_features=feature_cols,
                     outcome_name=target_col)
    m = dice_ml.Model(model=model, backend='sklearn',
                      model_type='classifier')
    return dice_ml.Dice(d, m, method='random')

print("Configuration loaded.")


# ─────────────────────────────────────────────
# Cell 2 | Case Study Generation
# ─────────────────────────────────────────────
all_cases = []

for d_key, d_info in DISEASE_REGISTRY.items():
    target        = d_info['target']
    label         = d_info['label']
    feature_cols  = d_info['feature_cols']
    cont_features = d_info['cont_features']

    print(f"\n{'='*60}")
    print(f"  Case Study — {label}")
    print(f"{'='*60}")

    df = pd.read_csv(
        os.path.join(BASE_OUT, f"{d_key}_total.csv")
    )
    for col in feature_cols:
        df[col] = df[col].astype(float)

    model  = joblib.load(
        os.path.join(BASE_MOD,
                     f"xgb_{d_key}_total.joblib")
    )
    X      = df[feature_cols].astype(float)
    y_pred = model.predict(X)
    y_prob = model.predict_proba(X)[:, 1]
    df['y_pred'] = y_pred
    df['y_prob'] = y_prob

    declined = df[df['y_pred'] == 1].copy()

    # Select representative cases:
    # Young: highest prob among young declined
    # Elderly: highest prob among elderly declined
    cases = {}
    for ag, ag_label in [
        ('young',   'Young (19–44)'),
        ('elderly', 'Elderly (65+)')
    ]:
        sub = declined[
            declined['age_group'] == ag
        ].sort_values('y_prob', ascending=False)
        if len(sub) > 0:
            cases[ag_label] = sub.iloc[0]

    exp = build_dice(df, feature_cols, target, model)

    for case_label, row in cases.items():
        print(f"\n  Case: {label} / {case_label}")
        print(f"  Predicted prob = {row['y_prob']:.4f}")

        query = make_query(row, feature_cols)

        # CFE without guardrail (N=5)
        cfe_ng = None
        try:
            dice_exp = exp.generate_counterfactuals(
                query, total_CFs=5,
                desired_class=0,
                random_seed=RANDOM_SEED,
                verbose=False
            )
            cfe_ng = dice_exp.cf_examples_list[0]\
                             .final_cfs_df
            if cfe_ng is not None and len(cfe_ng) > 0:
                cfe_ng = cfe_ng.iloc[0]
        except Exception:
            pass

        # CFE with guardrail (N=20, compliant only)
        cfe_wg = None
        try:
            dice_exp2 = exp.generate_counterfactuals(
                query, total_CFs=20,
                desired_class=0,
                random_seed=RANDOM_SEED,
                verbose=False
            )
            cfe_df2 = dice_exp2.cf_examples_list[0]\
                               .final_cfs_df
            if cfe_df2 is not None:
                for _, cr in cfe_df2.iterrows():
                    if (check_phi_caus(row, cr) and
                        check_phi_cost(
                            row, cr, cont_features) and
                        check_g1(cr, cont_features)):
                        cfe_wg = cr
                        break
        except Exception:
            pass

        # Build case table
        prob_ng = (
            model.predict_proba(
                pd.DataFrame(
                    [cfe_ng[feature_cols]]
                ).astype(float)
            )[0, 1]
            if cfe_ng is not None else np.nan
        )
        prob_wg = (
            model.predict_proba(
                pd.DataFrame(
                    [cfe_wg[feature_cols]]
                ).astype(float)
            )[0, 1]
            if cfe_wg is not None else np.nan
        )

        print(f"\n  {'Feature':<35} "
              f"{'Original':>10} "
              f"{'CFE (no guard)':>15} "
              f"{'CFE (guard)':>12}")
        print("  " + "-"*75)

        case_rows = []
        for feat in feature_cols:
            orig_val = row[feat]
            ng_val   = (cfe_ng[feat]
                        if cfe_ng is not None
                        else np.nan)
            wg_val   = (cfe_wg[feat]
                        if cfe_wg is not None
                        else np.nan)

            # Change flags
            ng_chg = ng_val - orig_val \
                     if not np.isnan(ng_val) else np.nan
            wg_chg = wg_val - orig_val \
                     if not np.isnan(wg_val) else np.nan

            feat_lbl = FEATURE_LABEL.get(feat, feat)

            # phi_cost check for display
            bound = PHI_COST_BOUNDS.get(feat)
            ng_ok = (abs(ng_chg) <= bound
                     if (bound and not np.isnan(ng_chg))
                     else None)
            wg_ok = (abs(wg_chg) <= bound
                     if (bound and not np.isnan(wg_chg))
                     else None)

            ng_flag = ('' if ng_ok is None
                       else '✅' if ng_ok else '❌')
            wg_flag = ('' if wg_ok is None
                       else '✅' if wg_ok else '❌')

            ng_str = (f"{ng_val:.1f} "
                      f"({ng_chg:+.1f}) {ng_flag}"
                      if not np.isnan(ng_val)
                      else 'N/A')
            wg_str = (f"{wg_val:.1f} "
                      f"({wg_chg:+.1f}) {wg_flag}"
                      if not np.isnan(wg_val)
                      else 'N/A')

            print(f"  {feat_lbl:<35} "
                  f"{orig_val:>10.1f} "
                  f"{ng_str:>25} "
                  f"{wg_str:>20}")

            case_rows.append({
                'Disease'      : label,
                'Case'         : case_label,
                'Feature'      : feat_lbl,
                'Original'     : round(orig_val, 2),
                'CFE_NoGuard'  : round(ng_val, 2)
                                 if not np.isnan(ng_val)
                                 else None,
                'CFE_Guard'    : round(wg_val, 2)
                                 if not np.isnan(wg_val)
                                 else None,
                'Delta_NoGuard': round(ng_chg, 2)
                                 if not np.isnan(ng_chg)
                                 else None,
                'Delta_Guard'  : round(wg_chg, 2)
                                 if not np.isnan(wg_chg)
                                 else None,
            })

        # Predicted prob row
        print(f"\n  {'Predicted probability':<35} "
              f"{row['y_prob']:>10.4f} "
              f"{prob_ng:>15.4f} "
              f"{prob_wg:>12.4f}")

        phi_caus_ng = (
            check_phi_caus(row, cfe_ng)
            if cfe_ng is not None else None
        )
        phi_cost_ng = (
            check_phi_cost(row, cfe_ng, cont_features)
            if cfe_ng is not None else None
        )
        phi_caus_wg = (
            check_phi_caus(row, cfe_wg)
            if cfe_wg is not None else None
        )
        phi_cost_wg = (
            check_phi_cost(row, cfe_wg, cont_features)
            if cfe_wg is not None else None
        )

        print(f"\n  phi_caus pass : "
              f"{'✅' if phi_caus_ng else '❌' if phi_caus_ng is False else 'N/A'} (no guard) | "
              f"{'✅' if phi_caus_wg else '❌' if phi_caus_wg is False else 'N/A'} (guard)")
        print(f"  phi_cost pass : "
              f"{'✅' if phi_cost_ng else '❌' if phi_cost_ng is False else 'N/A'} (no guard) | "
              f"{'✅' if phi_cost_wg else '❌' if phi_cost_wg is False else 'N/A'} (guard)")

        all_cases.extend(case_rows)

# Save
df_cases = pd.DataFrame(all_cases)
case_path = os.path.join(
    BASE_OUT, 'table_cfe_case_study.csv'
)
df_cases.to_csv(case_path, index=False)
print(f"\n\nCase study table saved : {case_path}")
print("\n=== 10_cfe_case_study.ipynb complete ===")

Configuration loaded.

  Case Study — Hypertension

  Case: Hypertension / Young (19–44)
  Predicted prob = 0.9284


100%|████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:18<00:00, 18.57s/it]



  Feature                               Original  CFE (no guard)  CFE (guard)
  ---------------------------------------------------------------------------
  Alcohol drinking frequency                 5.0               5.0 (+0.0)                   N/A
  Alcohol amount per occasion                4.0               4.0 (+0.0)                   N/A
  Current smoking                            1.0               1.0 (+0.0)                   N/A
  High-intensity PA: leisure                 2.0               2.0 (+0.0)                   N/A
  PA: transportation                         1.0               1.0 (+0.0)                   N/A
  Aerobic PA practice                        1.0               1.0 (+0.0)                   N/A
  Breakfast frequency                        1.0               1.0 (+0.0)                   N/A
  Health checkup (past year)                 1.0               1.0 (+0.0)                   N/A
  BMI (kg/m²)                               29.7             29.7 (+0.0) ✅ 

100%|████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:36<00:00, 36.02s/it]



  Feature                               Original  CFE (no guard)  CFE (guard)
  ---------------------------------------------------------------------------
  Alcohol drinking frequency                 6.0               6.0 (+0.0)                   N/A
  Alcohol amount per occasion                4.0               4.0 (+0.0)                   N/A
  Current smoking                            1.0               1.0 (+0.0)                   N/A
  High-intensity PA: leisure                 2.0               2.0 (+0.0)                   N/A
  PA: transportation                         2.0               1.4 (-0.6)                   N/A
  Aerobic PA practice                        0.0               0.0 (+0.0)                   N/A
  Breakfast frequency                        1.0               1.0 (+0.0)                   N/A
  Health checkup (past year)                 2.0               2.0 (+0.0)                   N/A
  BMI (kg/m²)                               27.8             27.8 (+0.0) ✅ 

...


  Feature                               Original  CFE (no guard)  CFE (guard)
  ---------------------------------------------------------------------------
  Alcohol drinking frequency                 1.0               1.0 (+0.0)                   N/A
  Alcohol amount per occasion                8.0               8.0 (+0.0)                   N/A
  Current smoking                            1.0               1.0 (+0.0)                   N/A
  High-intensity PA: leisure                 2.0               2.0 (+0.0)                   N/A
  PA: transportation                         2.0               2.0 (+0.0)                   N/A
  Aerobic PA practice                        0.0               0.0 (+0.0)                   N/A
  Breakfast frequency                        1.0               1.0 (+0.0)                   N/A
  Health checkup (past year)                 2.0               2.0 (+0.0)                   N/A
  BMI (kg/m²)                               24.3             24.3 (+0.0) ✅ 

100%|████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  3.92it/s]



  Feature                               Original  CFE (no guard)  CFE (guard)
  ---------------------------------------------------------------------------
  Alcohol drinking frequency                 2.0               2.0 (+0.0)                   N/A
  Alcohol amount per occasion                1.0               1.0 (+0.0)                   N/A
  Current smoking                            8.0               8.0 (+0.0)                   N/A
  High-intensity PA: leisure                 2.0               1.0 (-1.0)                   N/A
  PA: transportation                         2.0               2.0 (+0.0)                   N/A
  Aerobic PA practice                        0.0               0.0 (+0.0)                   N/A
  Breakfast frequency                        1.0               1.0 (+0.0)                   N/A
  Health checkup (past year)                 1.0               1.0 (+0.0)                   N/A
  BMI (kg/m²)                               28.8            13.6 (-15.1) ❌ 

100%|████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  3.70it/s]


  Feature                               Original  CFE (no guard)  CFE (guard)
  ---------------------------------------------------------------------------
  Alcohol drinking frequency                 1.0               1.0 (+0.0)                   N/A
  Alcohol amount per occasion                8.0               8.0 (+0.0)                   N/A
  Current smoking                            8.0               8.0 (+0.0)                   N/A
  High-intensity PA: leisure                 2.0               1.3 (-0.7)                   N/A
  PA: transportation                         2.0               2.0 (+0.0)                   N/A
  Aerobic PA practice                        0.0               0.0 (+0.0)                   N/A
  Breakfast frequency                        1.0               1.0 (+0.0)                   N/A
  Health checkup (past year)                 1.0               1.0 (+0.0)                   N/A
  BMI (kg/m²)                               33.2            18.2 (-15.0) ❌ 